# Análise de dados com SQL: Escola Online

Projeto de prática em SQL (PostgreSQL) usando um banco fictício de uma plataforma de cursos online. O objetivo é responder perguntas de negócio do tipo que um analista recebe no dia a dia: quem são os alunos, quais cursos vendem mais, quanto cada instrutor gera de receita e onde há evasão.

## O banco de dados

São 7 tabelas ligadas por chaves estrangeiras:

| Tabela | O que guarda |
|---|---|
| `categorias` | áreas dos cursos (Programação, Design etc.) |
| `instrutores` | professores, com cidade e data de contratação |
| `cursos` | catálogo, com categoria, instrutor, carga horária, preço e nível |
| `alunos` | cadastro dos alunos, com cidade e estado |
| `matriculas` | quem se matriculou em qual curso, com status e progresso |
| `pagamentos` | um pagamento por matrícula, com forma e status |
| `avaliacoes` | nota de 1 a 5, só para matrículas concluídas |

Relações: `cursos` aponta para `categorias` e `instrutores`; `matriculas` aponta para `alunos` e `cursos`; `pagamentos` e `avaliacoes` apontam para `matriculas`.

## Conferindo os dados

Antes das perguntas, uma contagem de linhas por tabela para garantir que o script foi carregado por completo.

In [ ]:
SELECT 'categorias' AS tabela, COUNT(*) AS linhas FROM categorias
UNION ALL
SELECT 'instrutores' AS tabela, COUNT(*) AS linhas FROM instrutores
UNION ALL
SELECT 'cursos' AS tabela, COUNT(*) AS linhas FROM cursos
UNION ALL
SELECT 'alunos' AS tabela, COUNT(*) AS linhas FROM alunos
UNION ALL
SELECT 'matriculas' AS tabela, COUNT(*) AS linhas FROM matriculas
UNION ALL
SELECT 'pagamentos' AS tabela, COUNT(*) AS linhas FROM pagamentos
UNION ALL
SELECT 'avaliacoes' AS tabela, COUNT(*) AS linhas FROM avaliacoes;

,tabela,linhas
0,categorias,5
1,instrutores,6
2,cursos,12
3,alunos,40
4,matriculas,90
5,pagamentos,90
6,avaliacoes,37


## Perguntas e consultas

### 1. Quais são os cursos de nível Iniciante, do mais barato ao mais caro?

In [ ]:

SELECT titulo, preco
FROM cursos
WHERE nivel = 'Iniciante'
ORDER BY preco;


,titulo,preco
0,Figma para Iniciantes,177.0
1,Fundamentos de UX Design,227.0
2,SQL Essencial,247.0
3,Marketing nas Redes Sociais,257.0
4,Finanças para Empreendedores,267.0
5,Python do Zero,297.0


São 6 cursos de nível Iniciante, com preços de R$ 177 (Figma para Iniciantes) a R$ 297 (Python do Zero).

### 2. Quantos alunos existem por estado?

In [ ]:

SELECT estado, COUNT(*) AS total_alunos
FROM alunos
GROUP BY estado
ORDER BY total_alunos DESC, estado;


,estado,total_alunos
0,RJ,14
1,SP,10
2,MG,5
3,RS,5
4,BA,3
5,PE,2
6,PR,1


Rio de Janeiro (14) e São Paulo (10) juntos têm 24 dos 40 alunos, ou 60% da base. Paraná aparece com um aluno só.

### 3. Qual é o preço médio dos cursos por categoria?

O nome da categoria está em outra tabela, então aqui entra o primeiro `JOIN`.


In [ ]:

SELECT categorias.nome AS categoria,
       ROUND(AVG(cursos.preco), 2) AS preco_medio
FROM cursos
INNER JOIN categorias ON cursos.id_categoria = categorias.id_categoria
GROUP BY categorias.nome
ORDER BY preco_medio DESC;


,categoria,preco_medio
0,Programação,372.0
1,Dados e Analytics,329.5
2,Negócios e Gestão,292.0
3,Marketing Digital,272.0
4,Design,202.0


Programação tem o maior preço médio (R$ 372,00) e Design o menor (R$ 202,00).

### 4. Quais alunos estão matriculados em "SQL Essencial"?

Consulta com três tabelas: `cursos` leva a `matriculas`, que leva a `alunos`.

In [ ]:

SELECT alunos.nome,
       alunos.cidade,
       matriculas.data_matricula,
       matriculas.status
FROM cursos
INNER JOIN matriculas ON cursos.id_curso = matriculas.id_curso
INNER JOIN alunos ON matriculas.id_aluno = alunos.id_aluno
WHERE cursos.titulo = 'SQL Essencial'
ORDER BY matriculas.data_matricula;


,nome,cidade,data_matricula,status
0,Pedro Almeida,Salvador,2022-07-30,Concluída
1,Henrique Araújo,Campinas,2022-09-17,Concluída
2,Bruno Silva,Duque de Caxias,2022-11-27,Concluída
3,Karen Barbosa,Rio de Janeiro,2022-12-29,Cancelada
4,Nicolas Carvalho,Duque de Caxias,2023-04-15,Em andamento
5,Camila Almeida,Duque de Caxias,2023-06-23,Concluída
6,Henrique Barbosa,Duque de Caxias,2023-07-03,Em andamento
7,Bruno Martins,Salvador,2023-09-04,Concluída
8,Mariana Carvalho,Salvador,2023-09-27,Concluída
9,Isabela Oliveira,São Paulo,2024-05-25,Concluída


São 10 matrículas, o maior número entre os 12 cursos. Em seguida vêm Power BI na Prática e Python para Análise de Dados, com 9 cada.

### 5. Qual é a receita total dos pagamentos com status "Pago"?

In [ ]:

SELECT COUNT(*) AS pagamentos_pagos,
       SUM(valor) AS receita_total
FROM pagamentos
WHERE status = 'Pago';


,pagamentos_pagos,receita_total
0,75,22438.9


Para ter o contexto completo, a mesma soma separada por status de pagamento:

In [ ]:

SELECT status,
       COUNT(*) AS quantidade,
       SUM(valor) AS total
FROM pagamentos
GROUP BY status
ORDER BY total DESC;


,status,quantidade,total
0,Pago,75,22438.9
1,Reembolsado,11,3092.5
2,Pendente,4,810.3


A receita efetiva é de R$ 22.438,90, em 75 pagamentos. Ficaram R$ 810,30 pendentes e R$ 3.092,50 foram reembolsados, o que equivale a cerca de 12% do valor de tudo que foi cobrado.

### 6. Quais cursos têm nota média acima de 4?


In [ ]:

SELECT cursos.titulo,
       ROUND(AVG(avaliacoes.nota), 2) AS nota_media,
       COUNT(avaliacoes.id_avaliacao) AS qtd_avaliacoes
FROM cursos
INNER JOIN matriculas ON cursos.id_curso = matriculas.id_curso
INNER JOIN avaliacoes ON matriculas.id_matricula = avaliacoes.id_matricula
GROUP BY cursos.titulo
HAVING AVG(avaliacoes.nota) > 4
ORDER BY nota_media DESC, cursos.titulo;


,titulo,nota_media,qtd_avaliacoes
0,Fundamentos de UX Design,5.00,3
1,Figma para Iniciantes,4.50,2
2,Finanças para Empreendedores,4.50,2
3,SQL Avançado com PostgreSQL,4.50,4
4,Gestão de Projetos Ágeis,4.33,3
5,Power BI na Prática,4.33,3


Seis cursos passam de 4. Fundamentos de UX Design tem a maior nota (5,00), mas com só 3 avaliações. Como cada curso tem entre 1 e 6 avaliações, uma única nota muda bastante a média, então o ranking deve ser lido com cuidado.

### 7. Quais alunos nunca se matricularam em nenhum curso?

In [ ]:

SELECT alunos.id_aluno, alunos.nome, alunos.cidade
FROM alunos
LEFT JOIN matriculas ON matriculas.id_aluno = alunos.id_aluno
WHERE matriculas.id_matricula IS NULL;


,id_aluno,nome,cidade
0,39,Isabela Araújo,Belo Horizonte
1,8,Ana Araújo,Niterói
2,23,Olivia Almeida,Campinas


São 3 alunos (7,5% da base): Ana Araújo, Olivia Almeida e Isabela Araújo. Todos os cursos têm pelo menos uma matrícula.

### 8. Qual é o ranking de instrutores por receita gerada?

In [ ]:

SELECT instrutores.nome,
       SUM(pagamentos.valor) AS receita
FROM instrutores
INNER JOIN cursos ON instrutores.id_instrutor = cursos.id_instrutor
INNER JOIN matriculas ON cursos.id_curso = matriculas.id_curso
INNER JOIN pagamentos ON matriculas.id_matricula = pagamentos.id_matricula
WHERE pagamentos.status = 'Pago'
GROUP BY instrutores.id_instrutor, instrutores.nome
ORDER BY receita DESC;


,nome,receita
0,Juliana Prado,5143.2
1,Carla Mendes,4761.0
2,Rafael Souza,4126.8
3,Paulo Teixeira,3155.3
4,Fernanda Lima,2751.3
5,Marcos Vieira,2501.3


Juliana Prado lidera, com R$ 5.143,20, seguida de Carla Mendes (R$ 4.761,00) e Rafael Souza (R$ 4.126,80). Marcos Vieira fica por último (R$ 2.501,30), e os dois cursos dele são os de menor preço do catálogo.

### 9. Qual é a taxa de conclusão por curso?

A taxa é o número de matrículas concluídas dividido pelo total de matrículas do curso. Para calcular isso em uma consulta só, usei `CASE WHEN`: cada matrícula concluída vale 1, as demais valem 0, e a média desses valores é a proporção de concluídas.

In [ ]:

SELECT cursos.titulo,
       COUNT(matriculas.id_matricula) AS matriculas,
       ROUND(AVG(CASE WHEN matriculas.status = 'Concluída' THEN 1.0 ELSE 0 END) * 100, 1) AS taxa_conclusao_pct
FROM cursos
INNER JOIN matriculas ON cursos.id_curso = matriculas.id_curso
GROUP BY cursos.titulo
ORDER BY taxa_conclusao_pct DESC;


,titulo,matriculas,taxa_conclusao_pct
0,Gestão de Projetos Ágeis,5,80.0
1,SQL Essencial,10,70.0
2,Python para Análise de Dados,9,66.7
3,Python do Zero,6,66.7
4,SQL Avançado com PostgreSQL,8,62.5
5,Marketing nas Redes Sociais,5,60.0
6,Google Ads e Tráfego Pago,7,57.1
7,Power BI na Prática,9,44.4
8,JavaScript Completo,7,42.9
9,Fundamentos de UX Design,8,37.5


Gestão de Projetos Ágeis tem a maior taxa (80%), seguido de SQL Essencial (70%) e dos dois cursos de Python (cerca de 67%). Figma para Iniciantes e Finanças para Empreendedores têm as menores (25% cada). A taxa considera todas as matrículas, inclusive as canceladas e as que ainda estão em andamento, então não mede só abandono.

## Resumo

- A base tem 40 alunos, 12 cursos e 90 matrículas; 60% dos alunos estão no RJ e em SP.
- A receita efetiva foi de R$ 22.438,90, com R$ 3.092,50 reembolsados e R$ 810,30 ainda pendentes.
- SQL Essencial é o curso com mais matrículas (10), e Gestão de Projetos Ágeis é o que mais conclui (80%).
- Três alunos se cadastraram mas nunca se matricularam, o que pode virar uma ação de reativação.
- Instrutores com cursos mais caros ou mais matriculados (Juliana Prado, Carla Mendes) concentram a receita.
